# 🤖 Data Agents Hands On Lab 🤖
In this lab we will explore the use of the `data_agent_run` function to trigger agents!

It opens up new possibilities like scheduled agents, event-driven agents, agents using other agents and more!

In [ ]:
%%sql -r setup_db
USE ROLE accountadmin;
CREATE DATABASE IF NOT EXISTS DATA_AGENT_HOL_DB;
CREATE SCHEMA IF NOT EXISTS DATA_AGENT_HOL_DB.HOL_SCHEMA;
USE DATA_AGENT_HOL_DB.HOL_SCHEMA;

## Step 1. Agent setup
First of all, let's set up a basic agent with some core capabilities. It will be able to query data about clinical trials from clinicaltrials.gov (available from the Marketplace!)

In order for it to query structured data, we need to create a semantic view

In [ ]:
%%sql -r setup_semantic_view
CREATE OR REPLACE SEMANTIC VIEW DATA_AGENT_HOL_DB.HOL_SCHEMA.CLINICAL_TRIALS_SV

    TABLES (
        clinical_trials AS CLINICAL_TRIALS_RESEARCH_DATABASE.CT.CLINICAL_TRIALS_SEARCH_VW
            PRIMARY KEY (NCT_ID)
            COMMENT = 'Clinical trial records from ClinicalTrials.gov'
    )

    DIMENSIONS (
        clinical_trials.trial_id AS NCT_ID
            COMMENT = 'The unique ClinicalTrials.gov identifier (e.g. NCT01234567)',
        clinical_trials.brief_title AS BRIEF_TITLE
            COMMENT = 'The brief human-readable title of the clinical trial',
        clinical_trials.study_url AS STUDY_URL
            COMMENT = 'Direct URL link to the trial on ClinicalTrials.gov',
        clinical_trials.overall_status AS OVERALL_STATUS
            COMMENT = 'Current recruitment status: RECRUITING, COMPLETED, NOT_YET_RECRUITING, ENROLLING_BY_INVITATION, ACTIVE_NOT_RECRUITING, TERMINATED, WITHDRAWN, SUSPENDED, UNKNOWN',
        clinical_trials.study_type AS STUDY_TYPE
            COMMENT = 'Type of study: INTERVENTIONAL, OBSERVATIONAL, EXPANDED_ACCESS',
        clinical_trials.phases AS PHASES
            COMMENT = 'Clinical trial phase: Phase 1, Phase 2, Phase 3, Phase 4, or NA',
        clinical_trials.conditions AS CONDITIONS
            COMMENT = 'Medical conditions or diseases being studied',
        clinical_trials.lead_sponsor AS LEAD_SPONSOR
            COMMENT = 'Organization or institution leading the trial',
        clinical_trials.sex AS SEX
            COMMENT = 'Eligible sex for participation: ALL, FEMALE, MALE',
        clinical_trials.update_date AS UPDATE_TS
            COMMENT = 'Timestamp when the trial record was last updated in Snowflake. Use this to find newly published or recently modified trials.'
    )

    METRICS (
        clinical_trials.trial_count AS COUNT(NCT_ID)
            COMMENT = 'Total number of clinical trials',
        clinical_trials.distinct_sponsor_count AS COUNT(DISTINCT LEAD_SPONSOR)
            COMMENT = 'Number of unique lead sponsors'
    )

    COMMENT = 'Semantic view over clinical trials for structured analytical queries';

### 1b. Create the Cortex Agent

The agent is configured with two tools:
- **cortex_search** — keyword-based retrieval of trial descriptions (uses the existing search service)
- **cortex_analyst_text_to_sql** — structured SQL queries (counts, filters by date, aggregations) powered by the semantic view

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT DATA_AGENT_HOL_DB.HOL_SCHEMA.CLINICAL_TRIALS_AGENT
    COMMENT = 'Agent that summarizes clinical trial updates using search and analyst tools'
    FROM SPECIFICATION
    $$
    models:
      orchestration: auto

    orchestration:
      budget:
        seconds: 120
        tokens: 16000

    instructions:
      response: "You are a clinical trials research assistant. Provide concise, data-driven summaries. When asked about recent or newly published trials, use the analyst tool to query by UPDATE_TS date and the search tool to retrieve trial details."
      orchestration: "For questions about counts, trends, or date-filtered data, use the analyst tool. For keyword searches on trial titles or topics, use the search tool. Combine both when appropriate."
      system: "You are an AI assistant that helps researchers monitor clinical trial publications. Always mention the number of trials found, key conditions studied, and notable sponsors when summarizing."

    tools:
      - tool_spec:
          type: "cortex_search"
          name: "trial_search"
          description: "Searches clinical trial brief titles and metadata including conditions, sponsors, phases, and status. Use this to find trials by topic, condition, sponsor, or keyword."
      - tool_spec:
          type: "cortex_analyst_text_to_sql"
          name: "trial_analyst"
          description: "Runs structured SQL queries over clinical trial data. Use this for counting trials, filtering by date (UPDATE_TS), aggregating by status, phase, sponsor, or conditions."

    tool_resources:
      trial_search:
        name: "CLINICAL_TRIALS_RESEARCH_DATABASE.CT.CLINICAL_TRIALS_SEARCH_SERVICE"
        max_results: 10
        title_column: "BRIEF_TITLE"
        id_column: "NCT_ID"
      trial_analyst:
        semantic_view: "DATA_AGENT_HOL_DB.HOL_SCHEMA.CLINICAL_TRIALS_SV"
        execution_environment:
          type: "warehouse"
          warehouse: "APP_WH"
    $$;

In [ ]:
%%sql -r verify_agent
SHOW AGENTS IN SCHEMA DATA_AGENT_HOL_DB.HOL_SCHEMA;

---
## Step 2: Call the Agent Using DATA_AGENT_RUN

`SNOWFLAKE.CORTEX.DATA_AGENT_RUN` takes two arguments:
1. The fully qualified agent name (string)
2. A JSON request body with `messages` (and optionally `thread_id`, `parent_message_id`)

Let's ask it to summarize the previous day's clinical trial updates.

In [ ]:
%%sql -r agent_summary_response
SELECT
    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
        'DATA_AGENT_HOL_DB.HOL_SCHEMA.CLINICAL_TRIALS_AGENT',
        $${
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Summarize the clinical trials that were updated in the most recent day available in the data. How many were there, and what conditions or topics do they cover?"
                        }
                    ]
                }
            ],
            "stream": false
        }$$
    ) AS AGENT_RESPONSE;

---
## Step 3: Parse the Agent Response

The agent returns a JSON string. Use `TRY_PARSE_JSON` to convert it to a VARIANT, then extract specific fields from the response structure.

In [ ]:
%%sql -r dataframe_2
SELECT index, value
FROM {{agent_summary_response}}, 
    LATERAL FLATTEN(input => TRY_PARSE_JSON(AGENT_RESPONSE):content) f

In [ ]:
%%sql -r dataframe_1
SELECT f.value:text::STRING AS ANSWER
FROM {{agent_summary_response}}, 
    LATERAL FLATTEN(input => TRY_PARSE_JSON(AGENT_RESPONSE):content) f
WHERE f.value:type = 'text'
ORDER BY index DESC
LIMIT 1;

---
## Step 4: Set Up a Snowflake Task That Calls the Agent

Create a task that runs once per day, calls the agent for a summary of the previous day's trial updates, and logs the result to a table.

In [ ]:
%%sql -r create_log_table
CREATE OR REPLACE TABLE DATA_AGENT_HOL_DB.HOL_SCHEMA.AGENT_LOG (
    LOG_ID INT AUTOINCREMENT,
    RUN_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    AGENT_QUESTION VARCHAR,
    AGENT_RAW_RESPONSE VARIANT,
    AGENT_TEXT_ANSWER VARCHAR
);

In [ ]:
%%sql -r create_alert
CREATE OR REPLACE TASK DATA_AGENT_HOL_DB.HOL_SCHEMA.DAILY_TRIAL_SUMMARY_TASK
    WAREHOUSE = DEFAULT_WH
    SCHEDULE = 'USING CRON 0 8 * * * America/New_York'
    AS
        INSERT INTO DATA_AGENT_HOL_DB.HOL_SCHEMA.AGENT_LOG
            (AGENT_QUESTION, AGENT_RAW_RESPONSE, AGENT_TEXT_ANSWER)
        WITH agent_result AS (
            SELECT
                TRY_PARSE_JSON(
                    SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
                        'DATA_AGENT_HOL_DB.HOL_SCHEMA.CLINICAL_TRIALS_AGENT',
                        $${
                            "messages": [
                                {
                                    "role": "user",
                                    "content": [
                                        {
                                            "type": "text",
                                            "text": "Summarize the clinical trials that were updated in the most recent day of the data. List the count by overall status and highlight the key conditions and brief titles."
                                        }
                                    ]
                                }
                            ],
                            "stream": false
                        }$$
                    )
                ) AS resp
        ),
        text_blocks AS (
            SELECT
                resp,
                f.value:text::STRING AS agent_text
            FROM agent_result,
                LATERAL FLATTEN(input => resp:content) f
            WHERE f.value:type::STRING = 'text'
            ORDER BY index DESC
            LIMIT 1
        )
        SELECT
            'Summarize the clinical trials that were updated in the most recent day of the data. List the count by overall status and highlight the key conditions and brief titles.',
            resp,
            agent_text
        FROM text_blocks;

### Resume the Task

Tasks are created in a **suspended** state by default. Resume it to start the schedule.

In [ ]:
ALTER TASK DATA_AGENT_HOL_DB.HOL_SCHEMA.DAILY_TRIAL_SUMMARY_TASK RESUME;
SHOW TASKS IN SCHEMA DATA_AGENT_HOL_DB.HOL_SCHEMA;

### Test the Task Manually

Use `EXECUTE TASK` to trigger it immediately rather than waiting for the schedule.

In [ ]:
%%sql -r execute_alert
EXECUTE TASK DATA_AGENT_HOL_DB.HOL_SCHEMA.DAILY_TRIAL_SUMMARY_TASK;

We can check whether our task ran successfully using the UI or the log table:

In [ ]:
%%sql -r task_history
SELECT
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    ERROR_CODE,
    ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_TRIAL_SUMMARY_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -1, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC;

In [ ]:
%%sql -r check_log
SELECT
    *
FROM DATA_AGENT_HOL_DB.HOL_SCHEMA.AGENT_LOG
ORDER BY RUN_TIMESTAMP DESC
LIMIT 5;

---
## Cleanup

Run the cell below to remove all objects created during this lab.

In [ ]:
%%sql -r cleanup
DROP DATABASE IF EXISTS DATA_AGENT_HOL_DB;